# Feature Selection

This notebook is for selecting candidate feature families without p-hacking the sign or keeping one-off same-sample winners. It uses every normalized date available under `MODL_WS_NORMALIZED_DIR`, labels partial dates, and separates per-date IC, pooled date-neutral IC, stability, redundancy, and final keep/quarantine/drop recommendations.

The output should be treated as a research filter. A feature is a candidate only when it has a plausible market reason, enough observations, stable sign, and acceptable redundancy.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

if "notebooks" not in sys.path:
    sys.path.append("notebooks")

from advanced_features import (
    FEATURE_DATASET_KEYS,
    available_dataset_dates,
    build_feature_set,
    build_targets,
    compute_ic_table,
    discover_datasets,
)

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))
HORIZONS = tuple(int(value) for value in os.environ.get("MODL_HORIZONS", "5,15,30").split(","))
DATE_FILTER = [value.strip() for value in os.environ.get("MODL_SELECT_DATES", "").split(",") if value.strip()]

INCLUDE_PARTIAL_DATES = True
MIN_OBS_PER_DATE = int(os.environ.get("MODL_MIN_OBS_PER_DATE", "30"))
MIN_TOTAL_OBS = int(os.environ.get("MODL_MIN_TOTAL_OBS", "50"))
MIN_ABS_IC_CANDIDATE = float(os.environ.get("MODL_MIN_ABS_IC_CANDIDATE", "0.10"))
MAX_KEEP_PER_FAMILY_TARGET = int(os.environ.get("MODL_MAX_KEEP_PER_FAMILY_TARGET", "3"))
REDUNDANCY_THRESHOLD = float(os.environ.get("MODL_REDUNDANCY_THRESHOLD", "0.90"))
SAVE_OUTPUTS = False

HARD_EXCLUDE_FEATURES = {"reference_price", "log_price"}

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 240)
pd.set_option("display.max_colwidth", 220)

ROOT, BAR_MINUTES, HORIZONS, INCLUDE_PARTIAL_DATES

## Date Inventory

Every normalized date is listed here. `is_full_dataset` means all datasets required by the full feature builder are present. Partial dates are included when `INCLUDE_PARTIAL_DATES = True`, but they remain flagged so stability summaries can be interpreted correctly.

In [ ]:
date_key_map = available_dataset_dates(ROOT)
date_inventory = pd.DataFrame(
    [
        {
            "date": date,
            "dataset_count": len(keys),
            "required_dataset_count": len(set(keys) & set(FEATURE_DATASET_KEYS)),
            "missing_required": sorted(set(FEATURE_DATASET_KEYS) - set(keys)),
            "is_full_dataset": set(FEATURE_DATASET_KEYS).issubset(keys),
        }
        for date, keys in sorted(date_key_map.items())
    ]
)
if DATE_FILTER:
    date_inventory = date_inventory[date_inventory["date"].isin(DATE_FILTER)].copy()
if not INCLUDE_PARTIAL_DATES:
    date_inventory = date_inventory[date_inventory["is_full_dataset"]].copy()

selected_dates = date_inventory["date"].tolist()
display(date_inventory)
selected_dates

## Build Feature And Target Panels

Each date is built independently so rolling features and targets do not leak across midnight boundaries. The combined panels use a `(date, minute)` index. Dates that fail to build are recorded in `build_errors` rather than silently dropped.

In [ ]:
feature_by_date: dict[str, pd.DataFrame] = {}
target_by_date: dict[str, pd.DataFrame] = {}
model_by_date: dict[str, pd.DataFrame] = {}
build_rows: list[dict[str, object]] = []
build_errors: list[dict[str, object]] = []

for date in selected_dates:
    date_tag = datetime.strptime(date, "%Y-%m-%d").strftime("%y-%m-%d")
    datasets = discover_datasets(ROOT, date_tag)
    try:
        feature_set = build_feature_set(datasets, horizons=HORIZONS, bar_minutes=BAR_MINUTES)
        features = feature_set.feature_matrix.copy()
        targets = build_targets(feature_set.reference_price, feature_set.term_structure, HORIZONS, bar_minutes=BAR_MINUTES)
        model = pd.concat([features, targets], axis=1).sort_index()

        feature_by_date[date] = features
        target_by_date[date] = targets
        model_by_date[date] = model
        build_rows.append(
            {
                "date": date,
                "is_full_dataset": set(FEATURE_DATASET_KEYS).issubset(date_key_map[date]),
                "datasets": len(datasets),
                "feature_rows": len(features),
                "feature_columns": features.shape[1],
                "target_rows": len(targets),
                "target_columns": targets.shape[1],
                "reference_obs": int(feature_set.reference_price.notna().sum()),
            }
        )
    except Exception as exc:  # noqa: BLE001 - notebook diagnostic should keep going
        build_errors.append({"date": date, "error_type": type(exc).__name__, "error": str(exc)})

build_summary = pd.DataFrame(build_rows).sort_values("date")
build_errors = pd.DataFrame(build_errors)

if not model_by_date:
    raise RuntimeError("No dates built successfully")

feature_panel = pd.concat(feature_by_date, names=["date", "minute"]).sort_index()
target_panel = pd.concat(target_by_date, names=["date", "minute"]).sort_index()
model_panel = pd.concat(model_by_date, names=["date", "minute"]).sort_index()

display(build_summary)
if not build_errors.empty:
    display(build_errors)
feature_panel.shape, target_panel.shape, model_panel.shape

## Feature Family Labels

Selection is reported by feature family first. Exact feature rankings are still shown, but family-level consistency matters more than a single column landing at the top of one target on one date.

In [ ]:
ROLLING_SUFFIXES = (
    f"_diff_{BAR_MINUTES}m",
    "_mean_5m",
    "_z_5m",
    "_mean_15m",
    "_z_15m",
    "_mean_30m",
    "_z_30m",
)


def base_feature_name(feature: str) -> str:
    for suffix in ROLLING_SUFFIXES:
        if feature.endswith(suffix):
            return feature[: -len(suffix)]
    return feature


def feature_family(feature: str) -> str:
    base = base_feature_name(feature)
    rules = [
        ("hawkes_bsi", ("hawkes_bsi_",)),
        ("trade_flow", ("trade_flow_imbalance_",)),
        ("trade_activity", ("trade_volume_", "trade_trade_count_", "trade_vwap_")),
        ("book_microstructure", ("book_",)),
        ("cross_venue", ("cross_",)),
        ("spread_mid_arbitrage", ("mid_", "spread_")),
        ("realized_vol", ("rv_", "bpv_", "jump_")),
        ("returns", ("ret_",)),
        ("term_structure", ("iv_", "term_", "short_iv_decimal")),
        ("option_smile", ("smile_",)),
        ("futures_basis", ("basis_",)),
        ("funding", ("estimated_funding_rate",)),
    ]
    for family, prefixes in rules:
        if base.startswith(prefixes):
            return family
    return "other"


def target_group(target: str) -> str:
    if target.startswith("target_future_return_"):
        return "future_return"
    if target.startswith("target_future_rv_"):
        return "future_rv"
    if target.startswith("target_vrp_"):
        return "vrp"
    if target.startswith("target_fir_return_"):
        return "fir_return"
    return "other"


numeric_feature_columns = sorted(
    column
    for column in feature_panel.select_dtypes(include=[np.number]).columns
    if column not in HARD_EXCLUDE_FEATURES and not column.startswith("target_")
)
target_columns = sorted(target_panel.columns)

feature_catalog = pd.DataFrame(
    {
        "feature": numeric_feature_columns,
        "base_feature": [base_feature_name(column) for column in numeric_feature_columns],
        "family": [feature_family(column) for column in numeric_feature_columns],
        "is_rolling_transform": [base_feature_name(column) != column for column in numeric_feature_columns],
    }
)
display(feature_catalog.groupby("family").size().rename("feature_count").sort_values(ascending=False).to_frame())
display(feature_catalog.head(20))
len(numeric_feature_columns), len(target_columns)

## Per-Date And Pooled IC

Per-date IC shows whether a relationship appears repeatedly. Pooled IC is computed after ranking values within each date, which reduces the chance that one volatile or partial day dominates the result. Positive and inverse IC are both valid; the notebook reports direction instead of filtering one side after the fact.

In [ ]:
per_date_tables: list[pd.DataFrame] = []

for date, model in model_by_date.items():
    feature_columns_for_date = [
        column
        for column in numeric_feature_columns
        if column in model.columns and model[column].notna().sum() >= MIN_OBS_PER_DATE
    ]
    target_columns_for_date = [
        column
        for column in target_columns
        if column in model.columns and model[column].notna().sum() >= MIN_OBS_PER_DATE
    ]
    ic = compute_ic_table(model, feature_columns_for_date, target_columns_for_date, min_obs=MIN_OBS_PER_DATE)
    if ic.empty:
        continue
    ic["date"] = date
    per_date_tables.append(ic)

if per_date_tables:
    per_date_ic = pd.concat(per_date_tables, ignore_index=True)
else:
    per_date_ic = pd.DataFrame(columns=["feature", "target", "n", "spearman_ic", "pearson_corr", "abs_ic", "date"])

ranked_model = (
    model_panel[numeric_feature_columns + target_columns]
    .replace([np.inf, -np.inf], np.nan)
    .groupby(level="date")
    .rank(pct=True)
)
pooled_ic = compute_ic_table(ranked_model, numeric_feature_columns, target_columns, min_obs=MIN_TOTAL_OBS)

for frame in [per_date_ic, pooled_ic]:
    if not frame.empty:
        frame["base_feature"] = frame["feature"].map(base_feature_name)
        frame["family"] = frame["feature"].map(feature_family)
        frame["target_group"] = frame["target"].map(target_group)
        frame["direction"] = np.where(frame["spearman_ic"] >= 0, "positive", "inverse")

print(f"per-date IC rows: {len(per_date_ic):,}")
print(f"pooled IC rows: {len(pooled_ic):,}")
display(per_date_ic.sort_values(["date", "abs_ic"], ascending=[True, False]).head(40))
display(pooled_ic.sort_values("abs_ic", ascending=False).head(40))

## Stability Summary

`days_with_ic` is the number of dates where the feature-target pair had enough observations. Small dates still build and remain in the pooled date-neutral panel, but they do not drive per-date stability unless they clear `MIN_OBS_PER_DATE`. `sign_consistency_share` is 1.0 when every eligible date agrees on sign. With the current data set, this table is mainly a guardrail against overreacting to one day.

In [ ]:
built_date_count = len(model_by_date)
min_stable_days = min(2, built_date_count)

if per_date_ic.empty:
    pair_stability = pd.DataFrame()
else:
    pair_stability = (
        per_date_ic.groupby(["feature", "target", "base_feature", "family", "target_group"], as_index=False)
        .agg(
            days_with_ic=("date", "nunique"),
            total_obs=("n", "sum"),
            mean_ic=("spearman_ic", "mean"),
            median_ic=("spearman_ic", "median"),
            mean_abs_ic=("abs_ic", "mean"),
            min_abs_ic=("abs_ic", "min"),
            max_abs_ic=("abs_ic", "max"),
            sign_sum=("spearman_ic", lambda values: np.sign(values).sum()),
        )
    )
    pair_stability["sign_consistency_share"] = pair_stability["sign_sum"].abs() / pair_stability["days_with_ic"]
    pair_stability["stable_sign"] = pair_stability["sign_consistency_share"] == 1.0
    pair_stability["mean_direction"] = np.where(pair_stability["mean_ic"] >= 0, "positive", "inverse")

pooled_for_merge = pooled_ic[["feature", "target", "spearman_ic", "abs_ic", "n"]].rename(
    columns={"spearman_ic": "pooled_rank_ic", "abs_ic": "pooled_abs_ic", "n": "pooled_obs"}
)
selection_pairs = pair_stability.merge(pooled_for_merge, on=["feature", "target"], how="outer")
for column in ["base_feature", "family", "target_group"]:
    if column not in selection_pairs or selection_pairs[column].isna().any():
        mapper = base_feature_name if column == "base_feature" else feature_family if column == "family" else target_group
        source_column = "feature" if column != "target_group" else "target"
        selection_pairs[column] = selection_pairs[column].fillna(selection_pairs[source_column].map(mapper))
selection_pairs["days_with_ic"] = selection_pairs["days_with_ic"].fillna(0).astype(int)
selection_pairs["pooled_abs_ic"] = selection_pairs["pooled_abs_ic"].fillna(0.0)
selection_pairs["sign_consistency_share"] = selection_pairs["sign_consistency_share"].fillna(0.0)
selection_pairs["selection_score"] = (
    selection_pairs[["pooled_abs_ic", "mean_abs_ic"]].max(axis=1).fillna(0.0)
    * np.sqrt(selection_pairs["days_with_ic"].clip(lower=1) / max(built_date_count, 1))
    * selection_pairs["sign_consistency_share"].clip(lower=0.25)
)

family_summary = (
    selection_pairs.groupby(["target_group", "family"], as_index=False)
    .agg(
        pairs=("feature", "count"),
        stable_pair_share=("stable_sign", "mean"),
        median_days_with_ic=("days_with_ic", "median"),
        median_mean_abs_ic=("mean_abs_ic", "median"),
        top_pooled_abs_ic=("pooled_abs_ic", "max"),
        top_selection_score=("selection_score", "max"),
    )
    .sort_values(["target_group", "top_selection_score"], ascending=[True, False])
)

display(selection_pairs.sort_values("selection_score", ascending=False).head(80))
display(family_summary)

## Selection Plots

The first plot shows which families have the strongest pooled date-neutral IC by target group. The second plot shows exact feature-target pairs, colored by family. These are diagnostics, not final proof.

In [ ]:
if not family_summary.empty:
    plt.figure(figsize=(15, 6))
    sns.barplot(
        data=family_summary,
        x="target_group",
        y="top_pooled_abs_ic",
        hue="family",
    )
    plt.title("Top Pooled Date-Neutral Abs IC By Feature Family")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()

    heatmap_data = family_summary.pivot_table(index="family", columns="target_group", values="top_pooled_abs_ic", aggfunc="max")
    plt.figure(figsize=(10, max(4, 0.35 * len(heatmap_data))))
    sns.heatmap(heatmap_data, cmap="mako", annot=True, fmt=".3f")
    plt.title("Family Strength By Target Group")
    plt.tight_layout()

top_pairs = selection_pairs.sort_values("selection_score", ascending=False).head(40).copy()
if not top_pairs.empty:
    top_pairs["feature_target"] = top_pairs["feature"] + " -> " + top_pairs["target"]
    plt.figure(figsize=(15, max(6, 0.25 * len(top_pairs))))
    sns.barplot(data=top_pairs, y="feature_target", x="selection_score", hue="family", dodge=False)
    plt.title("Top Exact Feature-Target Pairs By Stability-Adjusted Score")
    plt.xlabel("selection score")
    plt.ylabel("")
    plt.tight_layout()

## Redundancy Check

Highly correlated features should not all survive just because they each correlate with a target. Use this table to collapse redundant rolling variants or venue variants into the smallest defensible set.

In [ ]:
top_feature_names = (
    selection_pairs.sort_values("selection_score", ascending=False)
    .drop_duplicates("feature")
    .head(80)["feature"]
    .tolist()
)

if len(top_feature_names) >= 2:
    top_corr = ranked_model[top_feature_names].corr(method="pearson").abs()
    redundant_rows = []
    for left_idx, left in enumerate(top_corr.columns):
        for right in top_corr.columns[left_idx + 1 :]:
            corr = top_corr.loc[left, right]
            if pd.notna(corr) and corr >= REDUNDANCY_THRESHOLD:
                redundant_rows.append(
                    {
                        "feature_a": left,
                        "family_a": feature_family(left),
                        "feature_b": right,
                        "family_b": feature_family(right),
                        "abs_rank_corr": corr,
                    }
                )
    redundant_pairs = pd.DataFrame(redundant_rows).sort_values("abs_rank_corr", ascending=False)
else:
    redundant_pairs = pd.DataFrame(columns=["feature_a", "family_a", "feature_b", "family_b", "abs_rank_corr"])

display(redundant_pairs.head(80))
if len(top_feature_names) >= 2:
    plt.figure(figsize=(18, 14))
    sns.heatmap(top_corr, cmap="vlag", center=0, xticklabels=True, yticklabels=True)
    plt.title("Feature-Feature Absolute Rank Correlation: Top Selection Candidates")
    plt.tight_layout()

## Keep, Quarantine, Drop

`candidate_keep` means the feature-target pair passes the current mechanical gates and ranks inside the top capped set for its family and target group. `candidate_pool` passes the gates but is lower priority. `quarantine_more_data` is expected while the data set is small or partial. `drop_low_signal` means it currently lacks enough pooled or stable IC. Direction is shown explicitly; both positive and inverse relationships can be valid if they are stable out of sample.

In [ ]:
def decision_for_row(row: pd.Series) -> str:
    if row["feature"] in HARD_EXCLUDE_FEATURES:
        return "drop_leakage_or_level"
    if row["days_with_ic"] < min_stable_days:
        return "quarantine_more_data"
    if row["sign_consistency_share"] < 0.75:
        return "quarantine_unstable_sign"
    if max(row.get("pooled_abs_ic", 0.0), row.get("mean_abs_ic", 0.0) if pd.notna(row.get("mean_abs_ic", np.nan)) else 0.0) < MIN_ABS_IC_CANDIDATE:
        return "drop_low_signal"
    return "candidate_pool"


candidate_table = selection_pairs.copy()
candidate_table["decision"] = candidate_table.apply(decision_for_row, axis=1)
fallback_direction = pd.Series(
    np.where(candidate_table["pooled_rank_ic"] >= 0, "positive", "inverse"),
    index=candidate_table.index,
)
candidate_table["mean_direction"] = candidate_table["mean_direction"].fillna(fallback_direction)
pool_mask = candidate_table["decision"] == "candidate_pool"
candidate_table.loc[pool_mask, "family_target_rank"] = (
    candidate_table.loc[pool_mask]
    .groupby(["target_group", "family"])["selection_score"]
    .rank(method="first", ascending=False)
)
keep_mask = pool_mask & (candidate_table["family_target_rank"] <= MAX_KEEP_PER_FAMILY_TARGET)
candidate_table.loc[keep_mask, "decision"] = "candidate_keep"

decision_rank = {
    "candidate_keep": 0,
    "candidate_pool": 1,
    "quarantine_more_data": 2,
    "quarantine_unstable_sign": 3,
    "drop_low_signal": 4,
    "drop_leakage_or_level": 5,
}
candidate_table["decision_rank"] = candidate_table["decision"].map(decision_rank).fillna(99)

display_cols = [
    "decision",
    "family",
    "feature",
    "target",
    "target_group",
    "mean_direction",
    "days_with_ic",
    "sign_consistency_share",
    "mean_ic",
    "mean_abs_ic",
    "pooled_rank_ic",
    "pooled_abs_ic",
    "selection_score",
    "total_obs",
    "pooled_obs",
]
candidate_view = candidate_table.sort_values(["decision_rank", "selection_score"], ascending=[True, False])
display(candidate_view[display_cols].head(150))

feature_keep_candidates = (
    candidate_table[candidate_table["decision"] == "candidate_keep"]
    .sort_values("selection_score", ascending=False)
    .drop_duplicates("feature")
)
feature_quarantine_candidates = (
    candidate_table[candidate_table["decision"].str.startswith("quarantine")]
    .sort_values("selection_score", ascending=False)
    .drop_duplicates("feature")
)

FEATURE_KEEP_LIST = feature_keep_candidates["feature"].tolist()
FEATURE_QUARANTINE_LIST = feature_quarantine_candidates["feature"].head(120).tolist()

print(f"candidate_keep unique features: {len(FEATURE_KEEP_LIST)}")
print(f"quarantine unique features shown: {len(FEATURE_QUARANTINE_LIST)}")
FEATURE_KEEP_LIST[:80]

## Optional Save

Set `SAVE_OUTPUTS = True` to write the feature-selection tables to `MODL_WS_FEATURE_DIR/feature_select/bar_<N>m`. The notebook does not save by default.

In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "feature_select" / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    date_inventory.to_parquet(out_dir / "date_inventory.parquet")
    build_summary.to_parquet(out_dir / "build_summary.parquet")
    per_date_ic.to_parquet(out_dir / "per_date_ic.parquet")
    pooled_ic.to_parquet(out_dir / "pooled_rank_ic.parquet")
    selection_pairs.to_parquet(out_dir / "selection_pairs.parquet")
    family_summary.to_parquet(out_dir / "family_summary.parquet")
    candidate_table.to_parquet(out_dir / "candidate_table.parquet")
    redundant_pairs.to_parquet(out_dir / "redundant_pairs.parquet")
    print(f"wrote feature-selection outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")